# Day 14:  Advanced Transformer-Based Extraction


In [1]:
import sys

# Safely installing the HuggingFace transformers library 
!"{sys.executable}" -m pip install transformers


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: C:\Users\K Shridharan\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


### Importing libraries

In [2]:
from transformers import pipeline
import warnings
warnings.filterwarnings("ignore")

In [3]:
print("Downloading the internet's brain (BERT-NER)... this may take a minute!")
# Load a pre-trained BERT model that specializes in Named Entity Recognition
ner_transformer = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")
print("BERT is awake and ready!\n" + "="*40)

# The exact test sentence requested by your mentor
test_sentence = "Experience with Python, SQL and AWS."

print(f"Input: '{test_sentence}'\n")
print("Extracting entities...")
results = ner_transformer(test_sentence)

print("-" * 30)
for entity in results:
    # We format the output so it's easy to read
    print(f"Word: {entity['word']:10} | Guessed Category: {entity['entity_group']}")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

BERT is awake and ready!
Input: 'Experience with Python, SQL and AWS.'

Extracting entities...
------------------------------
Word: Python     | Guessed Category: MISC
Word: SQL        | Guessed Category: MISC
Word: AWS        | Guessed Category: MISC


## The Evolution of NLP in Skill Extraction

To solve the skill extraction problem, our architecture evolved through five stages:

1. **Traditional NLP (Rule-Based & Regex):** High precision for known terms, but zero flexibility for variations.
2. **Statistical NLP (TF-IDF & N-Grams):** Extracted frequent phrases across postings, but lacked semantic context.
3. **Word Embeddings (Sentence Transformers):** Captured conceptual similarity (e.g., mapping "ML" to "Machine Learning"), but suffered from boundary fuzziness.
4. **Custom Rule-Based NER (spaCy EntityRuler):** Provided structured extraction into custom entity categories (`SKILL`, `DATABASE`, `CLOUD_PLATFORM`).
5. **Contextual Transformers (BERT / DistilBERT / RoBERTa):** Evaluates the entire bidirectional context of text simultaneously using self-attention mechanisms.

### Core Transformer Architectures Overview
* **BERT (Bidirectional Encoder Representations from Transformers):** Reads sentences in both directions to establish full context.
* **DistilBERT:** A distilled, 40% smaller and 60% faster version of BERT retaining 97% of its language understanding.
* **RoBERTa (Robustly Optimized BERT Approach):** Trains BERT with dynamic masking over larger batches and longer sequences for improved generalization.
* **BERT-NER:** A BERT backbone fine-tuned with a token classification head on annotated entity datasets (e.g., CoNLL-2003).

# Advanced Option: Side-by-Side Model Comparison 
### (Pre-trained BERT vs Custom Skill Model)

In [4]:
import pandas as pd
import spacy

# 1. Setup Day 12 Custom NER pipeline for direct comparison
nlp_custom = spacy.load("en_core_web_sm")
ruler = nlp_custom.add_pipe("entity_ruler", before="ner")
ruler.add_patterns([
    {"label": "SKILL", "pattern": "Python"},
    {"label": "SKILL", "pattern": "SQL"},
    {"label": "CLOUD_PLATFORM", "pattern": "AWS"}
])



In [5]:
# 2. Process the test sentence through both pipelines
test_sentence = "Experience with Python, SQL and AWS."
doc_spacy = nlp_custom(test_sentence)
bert_output = ner_transformer(test_sentence)


In [6]:
# 3. Format into a side-by-side comparison table
bert_dict = {item['word'].strip(): item['entity_group'] for item in bert_output}
custom_dict = {ent.text: ent.label_ for ent in doc_spacy.ents}

comparison_data = []
for token in ["Python", "SQL", "AWS"]:
    comparison_data.append({
        "Token": token,
        "Pretrained BERT-NER Label": bert_dict.get(token, "NOT_DETECTED"),
        "Custom Pipeline Label": custom_dict.get(token, "NOT_DETECTED"),
        "Ideal Recruiter Label": "SKILL / CLOUD"
    })

df_comparison = pd.DataFrame(comparison_data)
print("Model Comparison on Test Input:")
display(df_comparison)

Model Comparison on Test Input:


,Token,Pretrained BERT-NER Label,Custom Pipeline Label,Ideal Recruiter Label
0,Python,MISC,SKILL,SKILL / CLOUD
1,SQL,MISC,SKILL,SKILL / CLOUD
2,AWS,MISC,CLOUD_PLATFORM,SKILL / CLOUD
